In [27]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split

In [28]:
DATA_PATH = "../data/raw/bitext_customer_support.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (26872, 5)


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [29]:
required_columns = [
    "flags",
    "instruction",
    "category",
    "intent",
    "response"
]

df = df[required_columns].copy()

print("Columns:")
print(df.columns.tolist())

df.head()

Columns:
['flags', 'instruction', 'category', 'intent', 'response']


,flags,instruction,category,intent,response
0,B,question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...
1,BQZ,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...
2,BLQZ,i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...
3,BL,I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...
4,BCELN,"I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...


In [30]:
missing_summary = pd.DataFrame({
    "Missing Count": df.isnull().sum(),
    "Missing Percentage": (df.isnull().mean() * 100).round(2)
})

missing_summary

,Missing Count,Missing Percentage
flags,0,0.0
instruction,0,0.0
category,0,0.0
intent,0,0.0
response,0,0.0


In [31]:
critical_columns = [
    "instruction",
    "category",
    "intent",
    "response"
]

before = len(df)

df = df.dropna(subset=critical_columns).copy()

after = len(df)

print("Rows removed because of missing critical values:", before - after)
print("Remaining rows:", after)

Rows removed because of missing critical values: 0
Remaining rows: 26872


In [32]:
text_columns = [
    "instruction",
    "category",
    "intent",
    "response"
]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

empty_counts = {
    col: (df[col] == "").sum()
    for col in text_columns
}

print("Empty values before removal:")
print(empty_counts)

for col in text_columns:
    df = df[df[col] != ""]

df = df.reset_index(drop=True)

print("\nShape after removing empty values:", df.shape)

Empty values before removal:
{'instruction': np.int64(0), 'category': np.int64(0), 'intent': np.int64(0), 'response': np.int64(0)}

Shape after removing empty values: (26872, 5)


In [33]:
exact_duplicates = df.duplicated().sum()

print("Exact duplicate rows:", exact_duplicates)

before = len(df)

df = df.drop_duplicates().reset_index(drop=True)

after = len(df)

print("Exact duplicate rows removed:", before - after)
print("Remaining rows:", after)

Exact duplicate rows: 0
Exact duplicate rows removed: 0
Remaining rows: 26872


In [34]:
duplicate_instruction_count = df["instruction"].duplicated().sum()

print("Repeated customer messages:", duplicate_instruction_count)

Repeated customer messages: 2237


In [35]:
instruction_label_check = (
    df.groupby("instruction")
    .agg(
        category_count=("category", "nunique"),
        intent_count=("intent", "nunique")
    )
)

conflicting_instructions = instruction_label_check[
    (instruction_label_check["category_count"] > 1) |
    (instruction_label_check["intent_count"] > 1)
]

print(
    "Customer messages with conflicting labels:",
    len(conflicting_instructions)
)

conflicting_instructions.head()

Customer messages with conflicting labels: 0


,category_count,intent_count
instruction,,


In [36]:
assert len(conflicting_instructions) == 0, (
    "Conflicting labels detected. Investigate before splitting."
)

In [37]:
def normalize_text(text):
    text = str(text)

    # Replace repeated spaces, tabs and line breaks with one space
    text = re.sub(r"\s+", " ", text)

    # Remove leading and trailing whitespace
    text = text.strip()

    return text


df["instruction_clean"] = df["instruction"].apply(normalize_text)

df[
    ["instruction", "instruction_clean"]
].sample(5, random_state=42)

,instruction,instruction_clean
9329,I can't talk with a human agent,I can't talk with a human agent
4160,I have got to locate hte bills from {{Person N...,I have got to locate hte bills from {{Person N...
18500,"I cannot pay, help me to inform of a problem w...","I cannot pay, help me to inform of a problem w..."
8840,I want help speaking to customer service,I want help speaking to customer service
5098,I try to see th accepted payment options,I try to see th accepted payment options


In [38]:
df["category"] = df["category"].str.strip()
df["intent"] = df["intent"].str.strip()

print("Unique categories:", df["category"].nunique())
print("Unique intents:", df["intent"].nunique())

Unique categories: 11
Unique intents: 27


In [39]:
unique_messages = (
    df[
        ["instruction", "intent", "category"]
    ]
    .drop_duplicates(subset=["instruction"])
    .reset_index(drop=True)
)

print("Total rows:", len(df))
print("Unique customer messages:", len(unique_messages))

Total rows: 26872
Unique customer messages: 24635


In [40]:
train_messages, temp_messages = train_test_split(
    unique_messages,
    test_size=0.30,
    random_state=42,
    stratify=unique_messages["intent"]
)

print("Train unique messages:", len(train_messages))
print("Temporary unique messages:", len(temp_messages))

Train unique messages: 17244
Temporary unique messages: 7391


In [41]:
validation_messages, test_messages = train_test_split(
    temp_messages,
    test_size=0.50,
    random_state=42,
    stratify=temp_messages["intent"]
)

print("Train:", len(train_messages))
print("Validation:", len(validation_messages))
print("Test:", len(test_messages))

Train: 17244
Validation: 3695
Test: 3696


In [42]:
train_instruction_set = set(
    train_messages["instruction"]
)

validation_instruction_set = set(
    validation_messages["instruction"]
)

test_instruction_set = set(
    test_messages["instruction"]
)

In [43]:
def assign_split(instruction):
    if instruction in train_instruction_set:
        return "train"

    if instruction in validation_instruction_set:
        return "validation"

    if instruction in test_instruction_set:
        return "test"

    return "unknown"


df["split"] = df["instruction"].apply(assign_split)

In [44]:
unknown_count = (df["split"] == "unknown").sum()

print("Unknown split rows:", unknown_count)

assert unknown_count == 0, "Some records were not assigned to a split."

Unknown split rows: 0


In [45]:
train_set = set(
    df.loc[df["split"] == "train", "instruction"]
)

validation_set = set(
    df.loc[df["split"] == "validation", "instruction"]
)

test_set = set(
    df.loc[df["split"] == "test", "instruction"]
)

train_validation_overlap = len(
    train_set.intersection(validation_set)
)

train_test_overlap = len(
    train_set.intersection(test_set)
)

validation_test_overlap = len(
    validation_set.intersection(test_set)
)

print("Train ∩ Validation:", train_validation_overlap)
print("Train ∩ Test:", train_test_overlap)
print("Validation ∩ Test:", validation_test_overlap)

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [46]:
assert train_validation_overlap == 0
assert train_test_overlap == 0
assert validation_test_overlap == 0

print("\nNo customer-message leakage detected.")


No customer-message leakage detected.


In [47]:
intent_distribution = pd.crosstab(
    df["intent"],
    df["split"],
    normalize="columns"
).mul(100).round(2)

intent_distribution

split,test,train,validation
intent,,,
cancel_order,3.44,3.76,3.78
change_order,4.09,3.64,3.68
change_shipping_address,3.64,3.61,3.63
check_cancellation_fee,3.56,3.53,3.53
check_invoice,3.96,3.68,3.68
check_payment_methods,3.74,3.71,3.73
check_refund_policy,3.74,3.70,3.71
complaint,3.74,3.72,3.73
contact_customer_service,3.74,3.72,3.73


In [48]:
missing_intents_per_split = (
    intent_split_check == 0
).sum()

missing_intents_per_split

split
test          0
train         0
validation    0
dtype: int64

In [49]:
intent_split_check = pd.crosstab(
    df["intent"],
    df["split"]
)

intent_split_check

split,test,train,validation
intent,,,
cancel_order,138,708,152
change_order,164,685,148
change_shipping_address,146,681,146
check_cancellation_fee,143,665,142
check_invoice,159,693,148
check_payment_methods,150,699,150
check_refund_policy,150,698,149
complaint,150,700,150
contact_customer_service,150,700,150


In [50]:
missing_intents_per_split = (
    intent_split_check == 0
).sum()

missing_intents_per_split

split
test          0
train         0
validation    0
dtype: int64

In [51]:
category_split_check = pd.crosstab(
    df["category"],
    df["split"]
)

category_split_check

split,test,train,validation
category,,,
ACCOUNT,895,4200,891
CANCEL,143,665,142
CONTACT,300,1399,300
DELIVERY,298,1411,285
FEEDBACK,300,1398,299
INVOICE,302,1399,298
ORDER,590,2799,599
PAYMENT,300,1398,300
REFUND,439,2093,460


In [52]:
df["instruction_word_count"] = (
    df["instruction"]
    .str.split()
    .str.len()
)

df["instruction_char_count"] = (
    df["instruction"]
    .str.len()
)

df["response_word_count"] = (
    df["response"]
    .str.split()
    .str.len()
)

df[
    [
        "instruction_word_count",
        "instruction_char_count",
        "response_word_count"
    ]
].describe()

,instruction_word_count,instruction_char_count,response_word_count
count,26872.000000,26872.000000,26872.000000
mean,8.690979,46.889513,104.789037
std,2.605004,10.897578,52.966204
min,1.000000,6.000000,9.000000
25%,7.000000,40.000000,72.000000
50%,9.000000,48.000000,90.000000
75%,11.000000,55.000000,124.000000
max,16.000000,92.000000,402.000000


In [53]:
df = df.reset_index(drop=True)

# Makes the cell safe if you accidentally run it twice
if "ticket_id" in df.columns:
    df = df.drop(columns=["ticket_id"])

df.insert(
    0,
    "ticket_id",
    [
        f"TICKET_{i:05d}"
        for i in range(1, len(df) + 1)
    ]
)

df[["ticket_id", "instruction"]].head()

,ticket_id,instruction
0,TICKET_00001,question about cancelling order {{Order Number}}
1,TICKET_00002,i have a question about cancelling oorder {{Or...
2,TICKET_00003,i need help cancelling puchase {{Order Number}}
3,TICKET_00004,I need to cancel purchase {{Order Number}}
4,TICKET_00005,"I cannot afford this order, cancel purchase {{..."


In [54]:
final_columns = [
    "ticket_id",
    "instruction",
    "instruction_clean",
    "category",
    "intent",
    "response",
    "flags",
    "instruction_word_count",
    "instruction_char_count",
    "response_word_count",
    "split"
]

In [55]:
missing_columns = [
    col for col in final_columns
    if col not in df.columns
]

print("Missing columns:", missing_columns)

Missing columns: []


In [56]:
assert len(missing_columns) == 0, (
    f"Missing required columns: {missing_columns}"
)

In [57]:
processed_df = df[final_columns].copy()

print("Processed dataset shape:", processed_df.shape)

processed_df.head()

Processed dataset shape: (26872, 11)


,ticket_id,instruction,instruction_clean,category,intent,response,flags,instruction_word_count,instruction_char_count,response_word_count,split
0,TICKET_00001,question about cancelling order {{Order Number}},question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...,B,6,48,37,train
1,TICKET_00002,i have a question about cancelling oorder {{Or...,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...,BQZ,9,58,48,test
2,TICKET_00003,i need help cancelling puchase {{Order Number}},i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...,BLQZ,7,47,205,train
3,TICKET_00004,I need to cancel purchase {{Order Number}},I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...,BL,7,42,166,train
4,TICKET_00005,"I cannot afford this order, cancel purchase {{...","I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...,BCELN,9,60,191,test


In [58]:
print("FINAL DATA QUALITY CHECK")
print("=" * 50)

print(f"Rows: {processed_df.shape[0]:,}")
print(f"Columns: {processed_df.shape[1]}")

print(
    "Missing values:",
    processed_df.isnull().sum().sum()
)

print(
    "Exact duplicate rows:",
    processed_df.duplicated().sum()
)

print(
    "Unique categories:",
    processed_df["category"].nunique()
)

print(
    "Unique intents:",
    processed_df["intent"].nunique()
)

print("\nSplit distribution:")
print(processed_df["split"].value_counts())

print("\nSplit percentages:")
print(
    processed_df["split"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

FINAL DATA QUALITY CHECK
Rows: 26,872
Columns: 11
Missing values: 0
Exact duplicate rows: 0
Unique categories: 11
Unique intents: 27

Split distribution:
split
train         18840
validation     4019
test           4013
Name: count, dtype: int64

Split percentages:
split
train         70.11
validation    14.96
test          14.93
Name: proportion, dtype: float64


In [59]:
OUTPUT_PATH = "../data/processed/cleaned_support_tickets.csv"

processed_df.to_csv(
    OUTPUT_PATH,
    index=False
)

print(f"Processed dataset saved to: {OUTPUT_PATH}")

Processed dataset saved to: ../data/processed/cleaned_support_tickets.csv


In [60]:
check_df = pd.read_csv(
    "../data/processed/cleaned_support_tickets.csv"
)

print("Saved dataset shape:", check_df.shape)

print("\nSplit distribution:")
print(check_df["split"].value_counts())

check_df.head()

Saved dataset shape: (26872, 11)

Split distribution:
split
train         18840
validation     4019
test           4013
Name: count, dtype: int64


,ticket_id,instruction,instruction_clean,category,intent,response,flags,instruction_word_count,instruction_char_count,response_word_count,split
0,TICKET_00001,question about cancelling order {{Order Number}},question about cancelling order {{Order Number}},ORDER,cancel_order,I've understood you have a question regarding ...,B,6,48,37,train
1,TICKET_00002,i have a question about cancelling oorder {{Or...,i have a question about cancelling oorder {{Or...,ORDER,cancel_order,I've been informed that you have a question ab...,BQZ,9,58,48,test
2,TICKET_00003,i need help cancelling puchase {{Order Number}},i need help cancelling puchase {{Order Number}},ORDER,cancel_order,I can sense that you're seeking assistance wit...,BLQZ,7,47,205,train
3,TICKET_00004,I need to cancel purchase {{Order Number}},I need to cancel purchase {{Order Number}},ORDER,cancel_order,I understood that you need assistance with can...,BL,7,42,166,train
4,TICKET_00005,"I cannot afford this order, cancel purchase {{...","I cannot afford this order, cancel purchase {{...",ORDER,cancel_order,I'm sensitive to the fact that you're facing f...,BCELN,9,60,191,test
